# 07 · Checks measure; gates decide

CHECK → METRIC → POLICY → GATE → PASS / WARN / FAIL.

The same 0.4% missing-customer result can PASS under one contract and FAIL under another. A gate is a pipeline decision, not an additional kind of validation rule.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Configurable policy and three decisions
Empty batches fail explicitly rather than dividing by zero. WARN allows publication in this teaching policy; production owners must document that choice.


In [ ]:
POLICY = dict(
    max_reject_percentage=1.0,
    max_null_customer_percentage=0.5,
    max_duplicate_orders=0,
    require_reconciliation=True,
)


def decide(metrics, policy=POLICY):
    failures = []
    if metrics["rows"] == 0:
        failures.append("empty input")
    if metrics["reject_pct"] > policy["max_reject_percentage"]:
        failures.append("reject percentage")
    if metrics["null_customer_pct"] > policy["max_null_customer_percentage"]:
        failures.append("customer completeness")
    if metrics["duplicate_orders"] > policy["max_duplicate_orders"]:
        failures.append("duplicate orders")
    if policy["require_reconciliation"] and not metrics["reconciled"]:
        failures.append("reconciliation")
    if failures:
        return "FAIL", failures
    near_limit = metrics["reject_pct"] > 0.8 * policy["max_reject_percentage"]
    return ("WARN" if near_limit else "PASS"), []


examples = [
    dict(
        rows=1000,
        reject_pct=0.0,
        null_customer_pct=0.0,
        duplicate_orders=0,
        reconciled=True,
    ),
    dict(
        rows=1000,
        reject_pct=0.9,
        null_customer_pct=0.2,
        duplicate_orders=0,
        reconciled=True,
    ),
    dict(
        rows=1000,
        reject_pct=3.0,
        null_customer_pct=0.8,
        duplicate_orders=2,
        reconciled=False,
    ),
]
for metrics in examples:
    print(metrics, decide(metrics))
assert [decide(m)[0] for m in examples] == ["PASS", "WARN", "FAIL"]


## Gates at different stages
RAW → schema/ingestion gate → BRONZE → validation gate → candidate SILVER → reconciliation/publication gate → GOLD. Rejected data and audit must survive failure. Candidate Silver is an inspection artifact and is not an approved published table.


In [ ]:
stages = [
    ("RAW", "required fields and parser health", "FAIL blocks Bronze"),
    (
        "BRONZE",
        "record metrics and duplicate policy",
        "FAIL blocks accepted publication",
    ),
    ("SILVER candidate", "counts, keys, control totals", "FAIL blocks Gold"),
]
spark.createDataFrame(stages, "stage string, checks string, effect string").show(
    truncate=False
)


## Do not make a dirty lesson pass by weakening the gate
Notebook 10 runs the intentionally dirty source under this strict policy, then runs a separate clean source under exactly the same policy. Failed runs retain their audit results and have no Gold publication manifest. Exercise: use a 0.7% reject metric and a stricter 0.5% limit; predict the result before running.
